# FastMCP — a multi-tool MCP server, a client, and an agent that picks between the tools

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhelamay/CPUT_ICAILY_AI2026_MaganLLM_WORKSHOP/blob/main/SA_LIB_WORKSHOP_Day2/09_mcp/09_mcp_fastmcp.ipynb)

**Author:** Dr. Nasser Mooman · Magan AI Inc. · nmooman@gmail.com · written with help from Claude (Anthropic)

> ⚠️ **Teaching code — do not use in production.** Short on purpose: no authentication, input validation, prompt-injection protection, error handling or tests. LLM answers can be wrong. Provided as is, without warranty.

**SA-LIB AI Workshop 2026 — Day 2: AI Agents**

**FastMCP** (`pip install fastmcp`) is the most popular high-level way to write MCP servers *and* clients in Python. `fastmcp_server.py` offers **6 tools** (math, units, text, time, weather, Wikipedia); `03` lists and calls them with FastMCP's `Client` (no LLM, no key); `04` is **one agent** that lets the LLM choose between them. At the end, the same server runs over **HTTP**.

Same example in every framework: **(1) first agent → (2) tools → (3) memory → (4) fallback → (5) multi-agent → complete agent**.

## 0. Setup
Install the packages (takes ~1 minute in Colab).

In [ ]:
%pip install -q "mcp>=2.0" "fastmcp>=4.0" "httpx" "openai" "python-dotenv" "ddgs"

### API keys
**Bring your own LLM API key** (Groq and Gemini have free tiers). No key is needed only if you already run a local server (Ollama, SGLang or vLLM): set `LLM_PROVIDER=ollama`, `OLLAMA_BASE_URL` and `LLM_MODEL`.

* **Colab:** click the 🔑 *Secrets* icon on the left, add e.g. `GROQ_API_KEY`, and enable notebook access.
* **VS Code / Jupyter / Linux:** put your keys in the `.env` file in the repo root (copy `.env.example`).
* Otherwise you'll be asked to paste a key below.

Choose the model provider with `LLM_PROVIDER`: `groq` · `gemini` · `openai` · `deepseek` · `openrouter` · `ollama`.

In [ ]:
import os, getpass
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))          # VS Code / Linux: read ../.env
try:                                           # Colab: read the Secrets panel
    from google.colab import userdata
    for key in ['GROQ_API_KEY', 'GOOGLE_API_KEY', 'OPENAI_API_KEY', 'DEEPSEEK_API_KEY', 'OPENROUTER_API_KEY', 'LANGSMITH_API_KEY', 'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_HOST', 'LLM_PROVIDER', 'LLM_MODEL']:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

os.environ.setdefault("LLM_PROVIDER", "groq")   # <- change the provider here if you like
provider = os.environ["LLM_PROVIDER"]
key_name = {"groq": "GROQ_API_KEY", "gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY",
            "deepseek": "DEEPSEEK_API_KEY", "openrouter": "OPENROUTER_API_KEY"}.get(provider)
if key_name and not os.getenv(key_name):
    os.environ[key_name] = getpass.getpass(f"{key_name}: ")
print("provider:", provider)

### Shared helper: `workshop_common.py`
Reads the provider settings, and holds the two tools every example uses: `web_search` (free DuckDuckGo) and `word_count`.

In [ ]:
%%writefile workshop_common.py
"""
workshop_common.py — shared helpers for every framework folder in this workshop.

(the SAME file is copied into each folder so every folder works on its own —
 edit shared/workshop_common.py and run `python tools/sync_common.py` to update the copies)

What it gives you:
  get_llm_config()   -> which provider / model / base_url / api key to use (read from .env)
  fallback_order()   -> the list of providers to try, in order, for the fallback examples
  ping()             -> quick "are you alive?" check against a provider
  web_search()       -> free DuckDuckGo search (no key)
  word_count()       -> the tiny "tool you wrote yourself" from the original guide

Every provider below speaks the OpenAI chat-completions protocol, which is why every
framework in this workshop can use every provider: we just change base_url + api_key + model.
"""

from __future__ import annotations

import os
from dataclasses import dataclass

from dotenv import find_dotenv, load_dotenv

# walks up from where you run python, so the .env in the repo root is always found
load_dotenv(find_dotenv(usecwd=True))

PROVIDERS = {
    #  name        base_url                                                     api-key env var       default model
    "groq":       ("https://api.groq.com/openai/v1",                           "GROQ_API_KEY",       "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY",     "gemini-2.5-flash"),
    "openai":     ("https://api.openai.com/v1",                                "OPENAI_API_KEY",     "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com",                                 "DEEPSEEK_API_KEY",   "deepseek-chat"),
    "openrouter": ("https://openrouter.ai/api/v1",                             "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),  None,                 "qwen2.5:7b"),
}


@dataclass
class LLMConfig:
    provider: str
    model: str
    base_url: str
    api_key: str

    @property
    def litellm_model(self) -> str:
        """Model name in LiteLLM format (used by CrewAI and Google ADK): 'openai/<model>'
        means 'an OpenAI-compatible endpoint' — combined with base_url it works for every provider."""
        return f"openai/{self.model}"


def get_llm_config(provider: str | None = None) -> LLMConfig:
    """Read LLM_PROVIDER / LLM_MODEL / <PROVIDER>_API_KEY from the environment."""
    provider = (provider or os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown LLM_PROVIDER '{provider}'. Choose one of: {', '.join(PROVIDERS)}")
    base_url, key_env, default_model = PROVIDERS[provider]
    api_key = os.getenv(key_env) if key_env else "ollama"  # ollama runs locally, no key
    if not api_key:
        raise RuntimeError(f"{key_env} is missing — add it to your .env file (see .env.example).")
    # LLM_MODEL only overrides the model of the MAIN provider, not of the fallbacks
    main_provider = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    model = (os.getenv("LLM_MODEL") if provider == main_provider else None) or default_model
    return LLMConfig(provider, model, base_url, api_key)


def fallback_order() -> list[str]:
    """Providers to try, in order. Default: your LLM_PROVIDER first, then LLM_FALLBACKS (groq,gemini)."""
    main = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    extra = [p.strip().lower() for p in os.getenv("LLM_FALLBACKS", "groq,gemini").split(",") if p.strip()]
    return list(dict.fromkeys([main, *extra]))  # remove duplicates, keep order


def ping(cfg: LLMConfig) -> None:
    """Send a 1-token request. Raises an exception if the provider is down, rate-limited or the key is wrong."""
    from openai import OpenAI

    OpenAI(base_url=cfg.base_url, api_key=cfg.api_key, timeout=20, max_retries=0).chat.completions.create(
        model=cfg.model, messages=[{"role": "user", "content": "ping"}], max_tokens=1
    )


def web_search(query: str) -> str:
    """Search the web with DuckDuckGo (free, no key) and return the top results as text."""
    try:
        from ddgs import DDGS

        results = DDGS().text(query, max_results=3)
        return "\n".join(f"- {r['title']}: {r['body']} ({r['href']})" for r in results) or "no results"
    except Exception as e:  # the free search is rate-limited: return the error instead of crashing
        return f"web search failed ({e}). try again in a few seconds."


def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())

### helper file: `fastmcp_server.py`
a MULTI-TOOL MCP server built with FastMCP (the `fastmcp` package: pip install fastmcp)

FastMCP is the most popular high-level way to write MCP servers in Python. one decorator,
@mcp.tool, turns a normal function into a tool any MCP client can discover and call.
the docstring becomes the tool description, the type hints become the input schema.

this server bundles 6 tools of different kinds, so you can see an agent CHOOSE between them:
  - calculate            pure Python (no internet)
  - convert_units        pure Python, with a fixed list of allowed values (Literal -> enum in the schema)
  - text_stats           pure Python, returns structured data (a dict -> JSON)
  - current_time         standard library (zoneinfo)
  - get_weather          public API: Open-Meteo (free, no key)
  - wikipedia_summary    public API: Wikipedia (free, no key)

this file is started BY the client (03_fastmcp_client.py / 04_fastmcp_agent.py) over stdio.
to serve it over HTTP instead: python 09_mcp/fastmcp_server.py http   (-> http://127.0.0.1:8000/mcp)

In [ ]:
%%writefile fastmcp_server.py
# a MULTI-TOOL MCP server built with FastMCP (the `fastmcp` package: pip install fastmcp)
#
# FastMCP is the most popular high-level way to write MCP servers in Python. one decorator,
# @mcp.tool, turns a normal function into a tool any MCP client can discover and call.
# the docstring becomes the tool description, the type hints become the input schema.
#
# this server bundles 6 tools of different kinds, so you can see an agent CHOOSE between them:
#   - calculate            pure Python (no internet)
#   - convert_units        pure Python, with a fixed list of allowed values (Literal -> enum in the schema)
#   - text_stats           pure Python, returns structured data (a dict -> JSON)
#   - current_time         standard library (zoneinfo)
#   - get_weather          public API: Open-Meteo (free, no key)
#   - wikipedia_summary    public API: Wikipedia (free, no key)
#
# this file is started BY the client (03_fastmcp_client.py / 04_fastmcp_agent.py) over stdio.
# to serve it over HTTP instead: python 09_mcp/fastmcp_server.py http   (-> http://127.0.0.1:8000/mcp)

import ast
import operator
import sys
from datetime import datetime
from typing import Literal
from zoneinfo import ZoneInfo

import httpx
from fastmcp import FastMCP

mcp = FastMCP("workshop-toolbox", instructions="General-purpose tools: math, units, text, time, weather, Wikipedia.")

HEADERS = {"User-Agent": "SA-LIB-workshop/1.0 (MCP demo)"}   # Wikipedia asks every client to identify itself


# ---------- 1. calculator (safe: we parse the expression, we never call eval) ----------
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.Mod: operator.mod, ast.USub: operator.neg, ast.UAdd: operator.pos}


def _eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        return OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
        return OPS[type(node.op)](_eval(node.operand))
    raise ValueError("only numbers and + - * / ** % ( ) are allowed")


@mcp.tool
def calculate(expression: str) -> float:
    """Evaluate a math expression exactly, e.g. '(1250 * 0.15) + 40' or '2 ** 10'. Use this instead of doing math in your head."""
    return _eval(ast.parse(expression, mode="eval").body)


# ---------- 2. unit converter ----------
TO_BASE = {"km": 1000, "m": 1, "mile": 1609.344, "ft": 0.3048,     # length -> metres
           "kg": 1, "lb": 0.45359237}                              # mass   -> kilograms


@mcp.tool
def convert_units(value: float, from_unit: Literal["km", "m", "mile", "ft", "kg", "lb", "C", "F"],
                  to_unit: Literal["km", "m", "mile", "ft", "kg", "lb", "C", "F"]) -> str:
    """Convert a value between units of length (km, m, mile, ft), mass (kg, lb) or temperature (C, F)."""
    if {from_unit, to_unit} <= {"C", "F"}:
        result = value if from_unit == to_unit else (value * 9 / 5 + 32 if from_unit == "C" else (value - 32) * 5 / 9)
    elif from_unit in TO_BASE and to_unit in TO_BASE and ({from_unit, to_unit} <= {"kg", "lb"}) == (from_unit in {"kg", "lb"}):
        result = value * TO_BASE[from_unit] / TO_BASE[to_unit]
    else:
        return f"cannot convert {from_unit} to {to_unit} (different kinds of unit)"
    return f"{value} {from_unit} = {round(result, 4)} {to_unit}"


# ---------- 3. text statistics (returns a dict: clients get it as structured JSON) ----------
@mcp.tool
def text_stats(text: str) -> dict:
    """Count the words, characters and sentences in a piece of text, and find the longest word."""
    words = text.split()
    return {"words": len(words), "characters": len(text),
            "sentences": sum(text.count(p) for p in ".!?") or 1,
            "longest_word": max(words, key=len) if words else ""}


# ---------- 4. current time in any time zone ----------
@mcp.tool
def current_time(timezone: str = "Africa/Tripoli") -> str:
    """Get the current date and time in an IANA time zone, e.g. 'Africa/Tripoli', 'Europe/London', 'Asia/Tokyo'."""
    try:
        now = datetime.now(ZoneInfo(timezone))
    except Exception:
        return f"unknown time zone '{timezone}'. use IANA names like 'Africa/Tripoli'."
    return now.strftime(f"%A %d %B %Y, %H:%M ({timezone})")


# ---------- 5. weather (public API, no key) ----------
@mcp.tool
def get_weather(city: str) -> str:
    """Get the current temperature, wind and humidity for a city, e.g. 'Tripoli' or 'Benghazi'."""
    found = httpx.get("https://geocoding-api.open-meteo.com/v1/search",
                      params={"name": city, "count": 1}, timeout=15).json().get("results")
    if not found:
        return f"I could not find a city called '{city}'."
    place = found[0]
    now = httpx.get("https://api.open-meteo.com/v1/forecast", timeout=15, params={
        "latitude": place["latitude"], "longitude": place["longitude"],
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m"}).json()["current"]
    return (f"{place['name']}, {place.get('country', '')}: {now['temperature_2m']} °C, "
            f"humidity {now['relative_humidity_2m']}%, wind {now['wind_speed_10m']} km/h")


# ---------- 6. Wikipedia summary (public API, no key) ----------
@mcp.tool
def wikipedia_summary(topic: str) -> str:
    """Get a short encyclopedia summary of a topic, place or person from Wikipedia."""
    title = topic.strip().replace(" ", "_")
    r = httpx.get(f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}",
                  headers=HEADERS, timeout=15, follow_redirects=True)
    if r.status_code != 200:
        return f"no Wikipedia page found for '{topic}'."
    data = r.json()
    return f"{data.get('title', topic)}: {data.get('extract', 'no summary available')}"


if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "http":
        mcp.run("http", host="127.0.0.1", port=8000)   # clients connect to http://127.0.0.1:8000/mcp
    else:
        mcp.run(show_banner=False, log_level="WARNING")   # default: stdio (the client starts us as a subprocess)

## 1. `03_fastmcp_client.py`

MCP 03 — FastMCP CLIENT, no LLM: connect to the multi-tool server, list its tools, call each one
same 4 steps as 01_mcp_client.py (start, handshake, list tools, call tools), but FastMCP's Client
does steps 1 + 2 for you: give it a path to a .py server and it starts it over stdio.

In [ ]:
import asyncio
import json
import sys
from pathlib import Path

from fastmcp import Client

# a file path -> stdio (the client starts the server); a URL -> HTTP (the server is already running)
URL = next((a for a in sys.argv[1:] if a.startswith("http")), None)
SERVER = URL or str(Path.cwd() / "fastmcp_server.py")

CALLS = [("calculate", {"expression": "(1250 * 0.15) + 40"}),
         ("convert_units", {"value": 100, "from_unit": "km", "to_unit": "mile"}),
         ("convert_units", {"value": 38, "from_unit": "C", "to_unit": "F"}),
         ("text_stats", {"text": "MCP is a standard plug for AI tools. Write once, use everywhere!"}),
         ("current_time", {"timezone": "Africa/Tripoli"}),
         ("get_weather", {"city": "Benghazi"}),
         ("wikipedia_summary", {"topic": "Leptis Magna"})]


async def main():
    async with Client(SERVER) as client:                      # 1 + 2: start/connect and handshake
        tools = await client.list_tools()                     # 3: discover
        print(f"{len(tools)} tools offered by the server:")
        for t in tools:
            print(f"  - {t.name}: {t.description}")
            print(f"      input schema: {json.dumps(t.input_schema.get('properties', {}))}")

        for name, args in CALLS:                              # 4: call
            result = await client.call_tool(name, args)
            print(f"\n>>> {name}({args})")
            print(result.data if result.data is not None else result.content[0].text)


if __name__ == "__main__":
    await main()

## 2. `04_fastmcp_agent.py`

MCP 04 — FastMCP CLIENT + ONE AGENT that picks from MANY tools
the agent gets all 6 tools from fastmcp_server.py and the LLM decides which ones to use —
often several in one question (weather + unit conversion + math ...).

  user --> [agent + LLM] --tool calls--> [FastMCP Client] --stdio--> [fastmcp_server.py] --> Open-Meteo / Wikipedia

In [ ]:
import asyncio
import json
from pathlib import Path

from fastmcp import Client
from openai import OpenAI

from workshop_common import get_llm_config

SERVER = str(Path.cwd() / "fastmcp_server.py")
cfg = get_llm_config()
llm = OpenAI(base_url=cfg.base_url, api_key=cfg.api_key)

SYSTEM = ("You are a helpful assistant. Use your tools for facts, time, weather and ALL arithmetic. "
          "You may call several tools. Be concise.")


def mcp_to_openai(tool) -> dict:
    """An MCP tool description -> the 'function calling' format every LLM provider understands."""
    return {"type": "function", "function": {
        "name": tool.name, "description": tool.description or "", "parameters": tool.input_schema}}


async def run_agent(client: Client, question: str, max_steps: int = 8) -> str:
    tools = [mcp_to_openai(t) for t in await client.list_tools()]              # discover tools at run time
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for _ in range(max_steps):                                                 # plan -> act -> observe
        msg = llm.chat.completions.create(model=cfg.model, messages=messages, tools=tools).choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg)
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            print(f"  [agent -> MCP] {call.function.name}({args})")
            result = await client.call_tool(call.function.name, args, raise_on_error=False)
            text = "\n".join(c.text for c in result.content if hasattr(c, "text"))
            messages.append({"role": "tool", "tool_call_id": call.id, "content": text})
    return "stopped: too many steps"


async def main():
    async with Client(SERVER) as client:
        for question in ["What time is it in Tripoli, and what's the weather there in Fahrenheit?",
                         "Tell me briefly about Leptis Magna. If I visit with 4 friends and a ticket costs 12.5 dinars, what is the total?",
                         "How many words are in this sentence: 'Agents choose tools, servers provide them'?"]:
            print(f"\nuser: {question}")
            print(f"agent: {await run_agent(client, question)}")


if __name__ == "__main__":
    await main()

## 3. The same server over HTTP
`stdio` only works when the client starts the server. With `http`, the server is a web service that any client (another machine, Claude Desktop, VS Code, ...) can reach at `http://127.0.0.1:8000/mcp`. We start it in the background, connect with the **same** `Client` (only the URL changes), then stop it.

In [ ]:
import subprocess, sys, time
from fastmcp import Client

server = subprocess.Popen([sys.executable, "fastmcp_server.py", "http"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)                                    # give the web server a moment to start
try:
    async with Client("http://127.0.0.1:8000/mcp") as client:
        print([t.name for t in await client.list_tools()])
        result = await client.call_tool("convert_units", {"value": 42, "from_unit": "km", "to_unit": "mile"})
        print(result.data)
finally:
    server.terminate()                            # stop the background server

---
*SA-LIB AI Workshop 2026 · based on [build-ai-agents-free](https://github.com/Moh4696/build-ai-agents-free) by m0h (MIT)*